# Libraries

In [1]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors
!pip install --upgrade transformers #FIX
!pip -q install optimum
!pip -q install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 73.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━

In [2]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [3]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [4]:
model_name = "EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [7]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [8]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_name},max_length=2048,dtype=float16,parallelize=True",
        "--tasks", cfg["task_name"],
        "--batch_size", str(3),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-28:12:25:33 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:12:25:39 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
2026-03-28:12:25:39 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (


{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-mon

INFO:httpx:HTTP Request: HEAD https://huggingface.co/EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[W328 12:25:59.465803317 Context.cpp:470] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/kernels-community/quantization-gptq/revision/main "HTTP/1.1 200 OK"
Fetching 0 files: 0it [00:00, ?it/s]
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/kernels-community/quantization-gptq/revision/main "HTTP/1.1 200 OK"
Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]INFO:httpx:HTTP Request: HEAD https://huggingface.co/kernels-community/quantization-gptq/resolve/4d9c20d702eb50ca44631773879001dc6b92ecbd/build/torch210-cxx11-cpu-x86_64-linux/__init__.py "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: H

{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  HFKernelLinear: loaded CPU gemm_4bit kernel from `kernels-community/quantization-gptq` variant `torch29-cxx11-cpu-x86_64-linux`.
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Kernel: Auto-selection: adding candidate `TritonV2QuantLinear`           
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
{'text/plain': '', 'text/html': '<pre style="margin:0; white-space:pre; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;"></pre>'}
INFO  Kernel: selected -> `TritonV2QuantLinea

...

hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.5837|±  |0.0062|
| - humanities                          |      2|none  |      |acc   |↑  |0.6400|±  |0.0129|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.4400|±  |0.0499|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.7200|±  |0.0451|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.7200|±  |0.0451|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.7200|±  |0.0451|
|  - international_law                  |   Finished mmlu

Sta

2026-03-28:12:50:34 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:12:50:39 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
2026-03-28:12:50:39 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attribut

Finished arc_challenge

Starting evaluation for: wikitext


2026-03-28:12:51:18 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-28:12:51:24 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
2026-03-28:12:51:24 WARNING  [evaluator:181] pretrained=EdgeCompress01/Llama-3.2-3B-Instruct-GPTQ-4bit appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).
Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e

Finished wikitext



# Save reports

In [9]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip